# SI Figure S2: literature Diels-Alder barrier predictions for ethylene + butadiene

Literature quantum-chemistry activation barriers for the ethylene + butadiene Diels-Alder reaction, spanning ~15-45 kcal/mol (experiment ~23.3 kcal/mol).

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import diels_alder

In [ ]:
def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
print(len(diels_alder.DATA), 'literature predictions; experimental barrier', diels_alder.EXPERIMENTAL_BARRIER, 'kcal/mol')
print('range:', min(e for _, e in diels_alder.DATA), 'to', max(e for _, e in diels_alder.DATA), 'kcal/mol')

In [ ]:
# One bar per literature prediction, grouped by method family (diels_alder.ordered_layout) and sorted by
# barrier within each group, colored by distance from the experimental value (red above, blue below),
# with a dashed line at the experimental barrier.
layout = diels_alder.ordered_layout(diels_alder.DATA)
methods, energies, positions = layout["methods"], layout["energies"], layout["positions"]

norm = TwoSlopeNorm(vmin=min(energies), vcenter=diels_alder.EXPERIMENTAL_BARRIER, vmax=max(energies))
colormap = plt.cm.RdBu_r
colors = [colormap(norm(e)) for e in energies]

fig, ax = plt.subplots(figsize=(16, 8))
ax.bar(positions, energies, color=colors, alpha=0.8, edgecolor="black", linewidth=0.5)
ax.axhline(y=diels_alder.EXPERIMENTAL_BARRIER, color="black", linestyle="--", linewidth=2)

# callout on the B3LYP/6-31G* bar at 23.1 kcal/mol (the level used elsewhere in the paper)
b3lyp = next((i for i, (m, e) in enumerate(zip(methods, energies))
              if "B3LYP/6-31G*" in m and abs(e - 23.1) < 0.1), None)
if b3lyp is not None:
    ax.annotate(f"Experimental: {diels_alder.EXPERIMENTAL_BARRIER} kcal/mol",
                xy=(positions[b3lyp], diels_alder.EXPERIMENTAL_BARRIER),
                xytext=(positions[b3lyp], diels_alder.EXPERIMENTAL_BARRIER + 8),
                arrowprops=dict(arrowstyle="->", color="black", lw=1),
                fontsize=10, fontweight="bold", ha="center",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.8))

ax.set_ylabel("Predicted barrier (kcal/mol)")
ax.set_xticks(positions)
ax.set_xticklabels(methods, rotation=45, ha="right", fontsize=8)
if b3lyp is not None:
    ax.get_xticklabels()[b3lyp].set_fontweight("bold")

y_range = ax.get_ylim()[1] - ax.get_ylim()[0]
label_y = ax.get_ylim()[0] - y_range * 0.25
for category, (start, end) in zip(layout["group_labels"], layout["group_boundaries"]):
    ax.text((start + end) / 2, label_y, category, ha="center", va="top",
            fontweight="bold", fontsize=11)

fig.tight_layout()
fig.savefig(figure_path("si_figure_s02_diels_alder.png"), dpi=300, bbox_inches="tight")
plt.show()